# 02 · Visualizações

**Projeto:** Futebol e economia · Visualização de Dados · Turma Bosch
**Equipe:** Luan Saviski, Miguel Gradella, Gabriel Wendt

Este notebook lê os dados limpos da Etapa 2 (`data/limpos/`) e responde a cada pergunta com um gráfico.
Cada gráfico segue o mesmo bloco: **pergunta → código → conclusão**, e é salvo em `images/`.

| # | Gráfico | Tipo | Biblioteca | Pergunta |
| --- | --- | --- | --- | --- |
| 01 | Resultado do país mais rico | Barras | matplotlib | 1 |
| 02 | Diferença de renda por resultado | Boxplot | seaborn | 2 |
| 03 | Vantagem do mais rico por Copa | Linha | matplotlib | 3 |
| 04 | Vitórias por país | Mapa | cartopy | 4 |
| 05 | Top 10 países em vitórias | Barras horizontais | matplotlib | 4 |
| 06 | IDH × gols por jogo | Dispersão com tendência | seaborn | 5 |
| 07 | Aproveitamento por faixa de IDH | Barras | matplotlib | 5 |
| 08 | Renda per capita das seleções | Histograma | seaborn | apoio |
| 09 | Correlação entre indicadores | Heatmap | seaborn | apoio |

**Cores:** azul é sempre o destaque (e a vitória do país mais rico); cinza é contexto.
O código de estilo (título, subtítulo, fonte e salvamento) fica em `src/graficos.py`.

## 0. Configuração

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.append(os.path.join(RAIZ, "src"))
from graficos import (AZUL, AZUL_CLARO, CINZA, CINZA_ESCURO, CORES_RESULTADO, NOMES_RESULTADO,
                      TEXTO, TEXTO_SUAVE, aplicar_estilo, fonte, num, salvar, titulo)

PASTA_LIMPOS = os.path.join(RAIZ, "data", "limpos")
PASTA_IMAGENS = os.path.join(RAIZ, "images")
aplicar_estilo()

jogos = pd.read_csv(os.path.join(PASTA_LIMPOS, "jogos_limpos.csv"))
selecoes = pd.read_csv(os.path.join(PASTA_LIMPOS, "selecoes_por_jogo.csv"))

# só jogos em que os dois países têm renda (perguntas 1 a 3)
jogos_renda = jogos.dropna(subset=["resultado_mais_rico"])

# seleção em cada Copa (ex.: Brasil 2002): unidade das perguntas 5 e dos gráficos de apoio
selecao_copa = (selecoes.groupby(["selecao", "iso3", "ano"])
                .agg(jogos=("vitoria", "size"), vitorias=("vitoria", "sum"),
                     gols_pro=("gols_pro", "mean"), gols_contra=("gols_contra", "mean"),
                     renda_pc=("renda_pc", "first"), idh=("idh", "first"))
                .reset_index())
selecao_copa["aproveitamento"] = selecao_copa["vitorias"] / selecao_copa["jogos"] * 100

print(f"{len(jogos)} jogos | {len(jogos_renda)} com renda dos dois países | "
      f"{len(selecao_copa)} participações de seleções")

## Gráfico 01 · Pergunta 1
**Em jogos de Copa (1990–2018), a seleção do país com maior renda per capita vence com mais frequência?**

In [ ]:
p1 = (jogos_renda["resultado_mais_rico"].value_counts(normalize=True)
      .mul(100).reindex(["vitoria", "empate", "derrota"]))

fig, ax = plt.subplots(figsize=(8, 4.2))
barras = ax.bar([NOMES_RESULTADO[r] for r in p1.index], p1.values,
                color=[CORES_RESULTADO[r] for r in p1.index], width=0.6)
ax.bar_label(barras, labels=[f"{v:.0f}%" for v in p1.values], padding=4,
             fontsize=13, fontweight="bold", color=TEXTO)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_ylim(0, p1.max() * 1.2)

titulo(fig, f"O país mais rico venceu {p1['vitoria']:.0f}% dos jogos e perdeu {p1['derrota']:.0f}%",
       f"Resultado da seleção do país com maior renda per capita (PPC), % dos {len(jogos_renda)} jogos de Copa, 1990–2018")
fonte(fig)
salvar(fig, "01_resultado_mais_rico", PASTA_IMAGENS)
plt.show()

razao = p1["vitoria"] / p1["derrota"]
print(f"Sugestão: Quando dois países de renda diferente se enfrentaram, o mais rico venceu "
      f"{p1['vitoria']:.0f}% das vezes e perdeu {p1['derrota']:.0f}% "
      f"({num(razao, 1)} vitória para cada derrota). Empates foram {p1['empate']:.0f}%.")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 02 · Pergunta 2
**A diferença de renda per capita entre os dois times é maior quando o mais rico vence do que quando empata ou perde?**

In [ ]:
ordem = ["vitoria", "empate", "derrota"]
dados2 = jogos_renda.assign(resultado=jogos_renda["resultado_mais_rico"].map(NOMES_RESULTADO),
                            dif_mil=jogos_renda["diferenca_renda"] / 1000)
medianas = dados2.groupby("resultado_mais_rico")["dif_mil"].median().reindex(ordem)

fig, ax = plt.subplots(figsize=(8, 4.6))
sns.boxplot(data=dados2, x="resultado", y="dif_mil", order=[NOMES_RESULTADO[r] for r in ordem],
            hue="resultado", palette={NOMES_RESULTADO[r]: CORES_RESULTADO[r] for r in ordem},
            width=0.5, fliersize=3, linewidth=1.2, legend=False, ax=ax)
for i, m in enumerate(medianas.values):
    ax.text(i + 0.3, m, f"mediana\nUS$ {m:.0f} mil", va="center", fontsize=9, color=TEXTO)
ax.set_xlabel("")
ax.set_ylabel("Diferença de renda per capita (mil US$ PPC)")

maior = "maior" if medianas["vitoria"] > medianas["derrota"] else "menor"
titulo(fig, f"Nas vitórias do mais rico, a distância de renda é {maior} do que nas derrotas",
       "Diferença de renda per capita entre as duas seleções, por resultado do país mais rico, Copas 1990–2018")
fonte(fig)
salvar(fig, "02_diferenca_renda_por_resultado", PASTA_IMAGENS)
plt.show()

print(f"Sugestão: A diferença mediana de renda foi de US$ {medianas['vitoria']:.0f} mil nas vitórias do mais rico, "
      f"US$ {medianas['empate']:.0f} mil nos empates e US$ {medianas['derrota']:.0f} mil nas derrotas. "
      f"(Se as caixas se sobrepõem muito, acrescente: a distância econômica sozinha não define o jogo.)")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 03 · Pergunta 3
**A vantagem do país mais rico mudou de 1990 a 2018?**

In [ ]:
p3 = (jogos_renda.groupby("ano")["resultado_mais_rico"]
      .apply(lambda s: (s == "vitoria").mean() * 100))
media3 = (jogos_renda["resultado_mais_rico"] == "vitoria").mean() * 100
inclinacao = np.polyfit(p3.index, p3.values, 1)[0] * 4   # pontos percentuais por Copa

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.axhline(media3, color=CINZA, linestyle="--", linewidth=1)
ax.text(p3.index.max() + 0.6, media3, f"média\n{media3:.0f}%", va="center", fontsize=9, color=TEXTO_SUAVE)
ax.plot(p3.index, p3.values, color=AZUL, linewidth=2.5, marker="o", markersize=7)
for ano, v in p3.items():
    ax.annotate(f"{v:.0f}%", (ano, v), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=9.5, color=TEXTO)
ax.set_xticks(p3.index)
ax.set_ylabel("Vitórias do país mais rico (% dos jogos)")
ax.set_ylim(0, max(p3.max(), media3) * 1.25)

direcao = "cresceu" if inclinacao > 1 else ("caiu" if inclinacao < -1 else "ficou estável")
titulo(fig, f"A vantagem do país mais rico {direcao} entre 1990 e 2018",
       "% dos jogos de cada Copa vencidos pela seleção do país com maior renda per capita")
fonte(fig)
salvar(fig, "03_vantagem_mais_rico_por_copa", PASTA_IMAGENS)
plt.show()

print(f"Sugestão: A taxa de vitórias do mais rico variou de {p3.min():.0f}% ({p3.idxmin()}) a {p3.max():.0f}% "
      f"({p3.idxmax()}). A tendência é de {'+' if inclinacao > 0 else ''}{num(inclinacao, 1)} pontos percentuais por Copa, então a vantagem {direcao}.")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 04 · Pergunta 4
**Onde estão os países que mais venceram jogos de Copa desde 1990?**

O mapa junta os dados pelo código `iso3`, que corresponde ao campo `ADM0_A3` do Natural Earth
(mais seguro que comparar nomes em inglês). Inglaterra, Escócia e Gales aparecem somados no Reino Unido.
Na primeira execução o cartopy baixa o contorno dos países: rodem com internet.

In [ ]:
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from matplotlib.colors import Normalize

vitorias_pais = selecoes.groupby("iso3")["vitoria"].sum()
maximo = vitorias_pais.max()
cmap = plt.cm.Blues
norma = Normalize(vmin=0, vmax=maximo)

fig = plt.figure(figsize=(11, 5.6))
ax = plt.axes(projection=ccrs.Robinson())
ax.set_global()
ax.spines["geo"].set_visible(False)

shp = shpreader.natural_earth("110m", "cultural", "admin_0_countries")
for pais in shpreader.Reader(shp).records():
    if pais.attributes["ADM0_A3"] == "ATA":   # Antártida não entra
        continue
    v = vitorias_pais.get(pais.attributes["ADM0_A3"], 0)
    cor = cmap(0.15 + 0.85 * norma(v)) if v > 0 else "#eeeeee"
    ax.add_geometries([pais.geometry], ccrs.PlateCarree(), facecolor=cor,
                      edgecolor="white", linewidth=0.4)

barra = fig.colorbar(plt.cm.ScalarMappable(norm=norma, cmap=cmap), ax=ax,
                     orientation="horizontal", fraction=0.035, pad=0.02, shrink=0.45)
barra.set_label("Vitórias em Copas (1990–2018)", color=TEXTO_SUAVE)
barra.outline.set_visible(False)

EUROPA_AMSUL = {"ALB","AUT","BEL","BIH","BGR","CHE","CZE","DEU","DNK","ESP","FRA","GBR","GRC","HRV","IRL",
                "ISL","ITA","NLD","NOR","POL","PRT","ROU","RUS","SRB","SVK","SVN","SWE","TUR","UKR",
                "ARG","BOL","BRA","CHL","COL","ECU","PRY","PER","URY","VEN"}
pct_eu_as = vitorias_pais[vitorias_pais.index.isin(EUROPA_AMSUL)].sum() / vitorias_pais.sum() * 100
titulo(fig, f"Europa e América do Sul somam {pct_eu_as:.0f}% das vitórias em Copas",
       "Total de jogos vencidos por país em Copas do Mundo, 1990–2018 · cinza = não venceu ou não jogou")
fonte(fig, "Fonte: Maven Analytics · World Cup · contornos: Natural Earth · Reino Unido soma Inglaterra, Escócia e Gales")
salvar(fig, "04_mapa_vitorias", PASTA_IMAGENS)
plt.show()

nome_pais = selecoes.groupby("iso3")["selecao"].agg(lambda s: s.mode()[0])
top3 = vitorias_pais.nlargest(3).rename(index=nome_pais)
print(f"Sugestão: {top3.index[0]}, {top3.index[1]} e {top3.index[2]} lideram com {top3.iloc[0]}, "
      f"{top3.iloc[1]} e {top3.iloc[2]} vitórias. Europa e América do Sul somam {pct_eu_as:.0f}% de todas as vitórias; "
      f"{(vitorias_pais > 0).sum()} países venceram ao menos um jogo.")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 05 · Pergunta 4 (ranking)
**Quais países mais venceram, e eles são ricos?**

O mapa mostra onde estão os vencedores; este ranking mostra os números e marca em azul os países
com renda per capita acima da mediana das seleções.

In [ ]:
ranking = (selecao_copa.groupby("iso3")
           .agg(selecao=("selecao", lambda s: s.mode()[0]), vitorias=("vitorias", "sum"),
                renda_media=("renda_pc", "mean"))
           .nlargest(10, "vitorias").sort_values("vitorias"))
mediana_renda = selecao_copa["renda_pc"].median()
cores5 = [AZUL if r > mediana_renda else CINZA for r in ranking["renda_media"]]

fig, ax = plt.subplots(figsize=(8, 5.2))
barras = ax.barh(ranking["selecao"], ranking["vitorias"], color=cores5, height=0.65)
ax.bar_label(barras, padding=4, fontsize=10.5, color=TEXTO)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="y", length=0)

ricos = sum(c == AZUL for c in cores5)
titulo(fig, f"{ricos} dos 10 países que mais venceram têm renda acima da mediana",
       "Vitórias em Copas, 1990–2018 · azul = renda per capita média acima da mediana das seleções")
fonte(fig)
salvar(fig, "05_top10_vitorias", PASTA_IMAGENS)
plt.show()

pobres = ranking[ranking["renda_media"] <= mediana_renda]["selecao"].tolist()
print(f"Sugestão: {ranking['selecao'].iloc[-1]} lidera com {ranking['vitorias'].iloc[-1]} vitórias. "
      f"{ricos} dos 10 maiores vencedores são países de renda acima da mediana; as exceções são {', '.join(pobres) or 'nenhuma'}.")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 06 · Pergunta 5
**Seleções de países com IDH maior marcam mais gols por jogo em Copas?**

Cada ponto é uma seleção em uma Copa (ex.: Brasil 2002). A linha é a tendência (regressão linear).

In [ ]:
d6 = selecao_copa.dropna(subset=["idh"])
r = np.corrcoef(d6["idh"], d6["gols_pro"])[0, 1]

fig, ax = plt.subplots(figsize=(8.5, 5))
sns.regplot(data=d6, x="idh", y="gols_pro", ax=ax,
            scatter_kws={"color": AZUL_CLARO, "s": 40, "alpha": 0.8, "edgecolor": "white"},
            line_kws={"color": AZUL, "linewidth": 2.5})
ax.set_xlabel("IDH do país no ano da Copa (0 a 1)")
ax.set_ylabel("Gols marcados por jogo")
ax.text(0.98, 0.95, f"correlação r = {num(r)}", transform=ax.transAxes, ha="right",
        fontsize=11, fontweight="bold", color=AZUL,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.9})

forca = ("forte" if abs(r) >= 0.5 else "moderada" if abs(r) >= 0.3 else "fraca")
sentido = "mais" if r > 0 else "menos"
titulo(fig, f"IDH maior se associa a {sentido} gols, mas a relação é {forca}",
       f"Gols por jogo de cada seleção em cada Copa × IDH do país, {len(d6)} participações, 1990–2018")
fonte(fig)
salvar(fig, "06_idh_x_gols", PASTA_IMAGENS)
plt.show()

complemento = ("Os pontos estão bem espalhados em torno da linha: o IDH ajuda pouco a prever quantos gols uma seleção marca."
               if abs(r) < 0.3 else "Os pontos acompanham a linha: países mais desenvolvidos tendem a marcar mais gols.")
print(f"Sugestão: A correlação entre IDH e gols por jogo é {num(r)} ({forca}). {complemento}")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 07 · Pergunta 5 (por faixas)
**O aproveitamento das seleções muda conforme a faixa de IDH do país?**

Dividimos as participações em quatro faixas de IDH com o mesmo número de seleções (quartis).

In [ ]:
d7 = selecao_copa.dropna(subset=["idh"]).copy()
d7["faixa"] = pd.qcut(d7["idh"], 4, labels=["1º quartil\n(IDH mais baixo)", "2º quartil",
                                            "3º quartil", "4º quartil\n(IDH mais alto)"])
p7 = d7.groupby("faixa", observed=True).agg(vitorias=("vitorias", "sum"), jogos=("jogos", "sum"))
p7["aproveitamento"] = p7["vitorias"] / p7["jogos"] * 100
cores7 = [AZUL if v == p7["aproveitamento"].max() else CINZA for v in p7["aproveitamento"]]

fig, ax = plt.subplots(figsize=(8.5, 4.4))
barras = ax.bar(p7.index.astype(str), p7["aproveitamento"], color=cores7, width=0.6)
ax.bar_label(barras, labels=[f"{v:.0f}%" for v in p7["aproveitamento"]], padding=4,
             fontsize=12, fontweight="bold", color=TEXTO)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_ylim(0, p7["aproveitamento"].max() * 1.2)

melhor = p7["aproveitamento"].idxmax().replace("\n", " ")
titulo(fig, f"As seleções do {melhor.split(' (')[0]} de IDH têm o maior aproveitamento",
       "% dos jogos vencidos pelas seleções em cada faixa de IDH do país, Copas 1990–2018")
fonte(fig)
salvar(fig, "07_aproveitamento_por_faixa_idh", PASTA_IMAGENS)
plt.show()

print("Sugestão: As seleções do quartil de IDH mais alto venceram "
      f"{p7['aproveitamento'].iloc[-1]:.0f}% dos jogos, contra {p7['aproveitamento'].iloc[0]:.0f}% "
      "no quartil mais baixo.")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 08 · Apoio
**Como se distribui a renda per capita dos países que disputaram as Copas?**

Ajuda a entender o contexto: as seleções da Copa são, em geral, de países ricos ou pobres?

In [ ]:
d8 = selecao_copa.dropna(subset=["renda_pc"])
mediana8 = d8["renda_pc"].median() / 1000

fig, ax = plt.subplots(figsize=(8.5, 4.4))
sns.histplot(d8["renda_pc"] / 1000, bins=20, color=AZUL_CLARO, edgecolor="white", kde=True,
             line_kws={"linewidth": 2}, ax=ax)
ax.lines[0].set_color(AZUL)
ax.axvline(mediana8, color=TEXTO, linestyle="--", linewidth=1)
ax.text(mediana8, ax.get_ylim()[1] * 0.95, f"  mediana: US$ {mediana8:.0f} mil", fontsize=9.5, color=TEXTO)
ax.set_xlabel("Renda per capita no ano da Copa (mil US$ PPC)")
ax.set_ylabel("Participações de seleções")

titulo(fig, f"Metade das seleções da Copa vem de países com renda acima de US$ {mediana8:.0f} mil",
       f"Distribuição da renda per capita dos países em {len(d8)} participações em Copas, 1990–2018")
fonte(fig)
salvar(fig, "08_distribuicao_renda", PASTA_IMAGENS)
plt.show()

print(f"Sugestão: A renda mediana dos países participantes é de US$ {mediana8:.0f} mil, "
      f"variando de US$ {d8['renda_pc'].min()/1000:.1f} mil a US$ {d8['renda_pc'].max()/1000:.0f} mil.")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

## Gráfico 09 · Apoio
**Quais indicadores andam juntos com o desempenho em campo?**

Correlação entre os indicadores econômicos e de desempenho, por seleção em cada Copa.
Azul = andam juntos; vermelho = andam em sentidos opostos; branco = sem relação.

In [ ]:
d9 = selecao_copa[["renda_pc", "idh", "gols_pro", "gols_contra", "aproveitamento"]].dropna()
d9.columns = ["Renda per capita", "IDH", "Gols marcados\npor jogo", "Gols sofridos\npor jogo", "Aproveitamento\n(%)"]
corr = d9.corr()

fig, ax = plt.subplots(figsize=(8.8, 6.2))
mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mascara, annot=True, fmt=".2f", cmap="RdBu", center=0, vmin=-1, vmax=1,
            linewidths=2, linecolor="white", cbar_kws={"shrink": 0.7, "label": "Correlação (r)"},
            annot_kws={"fontsize": 11}, ax=ax)
ax.tick_params(length=0)
plt.setp(ax.get_xticklabels(), rotation=0, fontsize=9.5)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9.5)

r_aprov = corr.loc["Renda per capita", "Aproveitamento\n(%)"]
titulo(fig, f"Renda e aproveitamento têm correlação de {num(r_aprov)}",
       "Correlação de Pearson por seleção em cada Copa, 1990–2018")
fonte(fig)
salvar(fig, "09_correlacao_indicadores", PASTA_IMAGENS)
plt.show()

r_renda_idh = corr.iloc[1, 0]
forca9 = "forte" if abs(r_aprov) >= 0.5 else "moderada" if abs(r_aprov) >= 0.3 else "fraca"
print(f"Sugestão: Renda e IDH andam juntos (r = {num(r_renda_idh)}). A correlação entre renda e aproveitamento é "
      f"{num(r_aprov)} ({forca9}), bem menor que a dos próprios gols com o aproveitamento.")

**Conclusão:** _(copie a frase sugerida impressa acima, confira se ela diz exatamente o que o gráfico mostra e ajuste com suas palavras)_

---
## Checklist da Etapa 3

- [ ] O notebook roda do início ao fim (*Run All*) sem erro
- [ ] 9 gráficos de 7 tipos (barras, boxplot, linha, mapa, barras horizontais, dispersão, histograma, heatmap)
- [ ] 4 com seaborn (02, 06, 08, 09) e 1 mapa com cartopy (04)
- [ ] Toda pergunta da Etapa 1 tem gráfico (P1: 01 · P2: 02 · P3: 03 · P4: 04 e 05 · P5: 06 e 07)
- [ ] Cada conclusão escrita em Markdown, conferida com o gráfico
- [ ] 9 PNGs em `images/`